In [ ]:
"""
Two-panel scalability chart comparing gridr vs scipy across interpolation methods:

  Panel A (left)  — Absolute execution time vs n (log-log scale)
                    One line per method × adapter, with IQR band
  Panel B (right) — Relative speedup ratio scipy/gridr vs n
                    ratio > 1  → gridr faster (green zone)
                    ratio < 1  → scipy faster (red zone)

Note : parse_result has to be executed first on json benchmark output
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

# ── Design tokens ─────────────────────────────────────────────────────────────
BG      = "#0b0f1a"
SURFACE = "#111827"
CARD    = "#16213a"
BORDER  = "#1f2d45"
TEXT    = "#e2e8f0"
MUTED   = "#64748b"
GOLD    = "#fbbf24"
GREEN   = "#4ade80"
RED     = "#f87171"

# One color per method — consistent across both panels
METHOD_COLORS = {
    "nearest":  "#38bdf8",   # sky
    "linear":   "#a78bfa",   # violet
    "bspline3": "#34d399",   # emerald
    "bspline5": "#fb923c",   # amber
    "bspline7": "#f472b6",   # pink
}
METHOD_LABELS = {
    "nearest":  "Nearest",
    "linear":   "Linear",
    "bspline3": "B-Spline 3",
    "bspline5": "B-Spline 5",
    "bspline7": "B-Spline 7",
}

GRIDR_LS = "-"        # solid   → gridr
SCIPY_LS = "--"       # dashed  → scipy
GRIDR_MK = "o"        # circle  → gridr
SCIPY_MK = "s"        # square  → scipy

METHODS = ["nearest", "linear", "bspline3", "bspline5", "bspline7"]
SIZES   = [100, 500, 1000, 2000, 4000]


_COLS = ["n", "method", "adapter", "time_min", "time_q1",
         "time_median", "time_q3", "time_max", "time_stddev"]


# ── Data helpers ──────────────────────────────────────────────────────────────

def _prepare(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize a parsed DataFrame (seconds) to milliseconds.
    Accepts both load_benchmark_json output and the bundled sample format.
    """
    df = df.copy()
    time_cols = ["time_min", "time_q1", "time_median",
                 "time_q3", "time_max", "time_stddev"]
    for col in time_cols:
        if col in df.columns:
            df[col] = df[col] * 1000.0   # s → ms
    return df


def _series(df: pd.DataFrame, method: str, adapter: str) -> pd.DataFrame:
    """
    Extract the time series for a (method, adapter) pair, sorted by n.
    Returns a DataFrame with columns [n, median, q1, q3, min, max].
    """
    mask = (df["method"] == method) & (df["adapter"] == adapter)
    sub  = df[mask].sort_values("n")
    return sub[["n", "time_median", "time_q1", "time_q3",
                "time_min", "time_max"]].rename(columns={
        "time_median": "median",
        "time_q1":     "q1",
        "time_q3":     "q3",
        "time_min":    "lo",
        "time_max":    "hi",
    })


def _ratio_series(df: pd.DataFrame, method: str) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute scipy_median / gridr_median for each n where both are available.
    Returns (sizes_array, ratios_array).
    """
    g = _series(df, method, "gridr")
    s = _series(df, method, "scipy")

    merged = g.merge(s, on="n", suffixes=("_g", "_s"))
    ratios = merged["median_s"] / merged["median_g"]
    return merged["n"].values, ratios.values


# ── Axis styling helpers ──────────────────────────────────────────────────────

def _style_ax(ax: plt.Axes) -> None:
    """Apply common dark-theme styling to an axis."""
    ax.set_facecolor(CARD)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(BORDER)
    ax.spines["bottom"].set_color(BORDER)
    ax.tick_params(colors=MUTED, labelsize=9, which="both")
    ax.xaxis.label.set_color(MUTED)
    ax.yaxis.label.set_color(MUTED)
    ax.grid(color=BORDER, linewidth=0.6, alpha=0.7, which="major")
    ax.grid(color=BORDER, linewidth=0.3, alpha=0.4, which="minor")


def _fmt_ms(ms: float, _pos) -> str:
    """Y-axis tick formatter for the absolute panel."""
    if ms < 1.0:
        return f"{ms * 1000:.0f}µs"
    if ms < 1000.0:
        return f"{ms:.0f}ms"
    return f"{ms / 1000:.1f}s"


# ── Main plotting function ────────────────────────────────────────────────────

def plot_scalability(
    df: pd.DataFrame | None = None,
    methods: list[str] | None = None,
    output:  str = None,
    fmt:     str = "png",
    dpi:     int = 300,
) -> plt.Figure:
    """
    Generate a two-panel scalability figure.

    Panel A  Absolute median execution time vs n — log-log scale.
             Solid lines = gridr, dashed = scipy.
             Shaded IQR band around each line.

    Panel B  Relative speedup scipy/gridr vs n — linear scale.
             Horizontal reference at ratio=1 (parity line).
             Green zone = gridr faster, red zone = scipy faster.

    Parameters
    ----------
    df      : parsed benchmark DataFrame (load_benchmark_json output).
              If None, the bundled sample data is used.
    methods : methods to include (default: all five).
    output  : output file path.
    fmt     : 'png' or 'svg'.
    dpi     : resolution for PNG (300 recommended for PowerPoint).
    """
    if df is None:
        df = _sample_df()
    df = _prepare(df)

    methods = methods or METHODS

    # ── Figure & axes ─────────────────────────────────────────────────────────
    fig, (ax_abs, ax_rel) = plt.subplots(
        1, 2,
        figsize=(16, 7),
        facecolor=BG,
        gridspec_kw={"width_ratios": [1.1, 1], "wspace": 0.35},
    )
    fig.patch.set_facecolor(BG)
    _style_ax(ax_abs)
    _style_ax(ax_rel)

    # ── Panel A — Absolute, log-log ───────────────────────────────────────────
    ax_abs.set_xscale("log")
    ax_abs.set_yscale("log")

    for method in methods:
        color = METHOD_COLORS[method]
        label = METHOD_LABELS[method]

        g = _series(df, method, "gridr")
        s = _series(df, method, "scipy")

        if not g.empty:
            ax_abs.plot(
                g["n"], g["median"],
                color=color, linewidth=2.2,
                linestyle=GRIDR_LS, marker=GRIDR_MK,
                markersize=7, markeredgewidth=1.5,
                markeredgecolor=BG, zorder=4,
                label=f"{label}  gridr",
            )
            ax_abs.fill_between(
                g["n"], g["q1"], g["q3"],
                color=color, alpha=0.15, zorder=2,
            )

        if not s.empty:
            ax_abs.plot(
                s["n"], s["median"],
                color=color, linewidth=2.2,
                linestyle=SCIPY_LS, marker=SCIPY_MK,
                markersize=7, markeredgewidth=1.5,
                markeredgecolor=BG, zorder=4,
                label=f"{label}  scipy",
            )
            ax_abs.fill_between(
                s["n"], s["q1"], s["q3"],
                color=color, alpha=0.10, zorder=2,
                linestyle=":",
            )

    ax_abs.set_xlabel("Grid size n", fontsize=10, labelpad=8)
    ax_abs.set_ylabel("Execution time", fontsize=10, labelpad=8)
    ax_abs.set_title(
        "Absolute execution time  (log scale)",
        fontsize=12, fontweight="bold",
        color=TEXT, pad=14,
    )
    ax_abs.yaxis.set_major_formatter(mticker.FuncFormatter(_fmt_ms))
    ax_abs.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"{int(x):,}")
    )
    ax_abs.set_xticks(SIZES)
    ax_abs.set_xticklabels([f"{n:,}" for n in SIZES], fontsize=8.5)

    # Annotation: slope guides (O(n) and O(n²)) for reference
    n_arr = np.array([SIZES[0], SIZES[-1]], dtype=float)
    y_ref = df[(df["method"] == "nearest") & (df["adapter"] == "gridr")][
        "time_median"].min() * 0.5
    #ax_abs.plot(
    #    n_arr, y_ref * (n_arr / n_arr[0]),
    #    color=MUTED, linewidth=0.8, linestyle=":", alpha=0.5, zorder=1,
    #)
    #ax_abs.text(
    #    SIZES[-1] * 1.02, y_ref * (SIZES[-1] / SIZES[0]) * 1.1,
    #    "O(n)", fontsize=7.5, color=MUTED, va="center",
    #    fontfamily="monospace",
    #)

    # Legend — split into two columns: one per adapter style
    gridr_patch = plt.Line2D(
        [], [], color="white", linewidth=2, linestyle=GRIDR_LS,
        marker=GRIDR_MK, markersize=6, label="── gridr",
    )
    scipy_patch = plt.Line2D(
        [], [], color="white", linewidth=2, linestyle=SCIPY_LS,
        marker=SCIPY_MK, markersize=6, label="- - scipy",
    )
    method_handles = [
        plt.Line2D([], [], color=METHOD_COLORS[m], linewidth=2.5,
                   label=METHOD_LABELS[m])
        for m in methods
    ]
    ax_abs.legend(
        handles=[gridr_patch, scipy_patch, *method_handles],
        loc="upper left",
        frameon=True,
        framealpha=0.25,
        facecolor=SURFACE,
        edgecolor=BORDER,
        labelcolor=TEXT,
        fontsize=8.5,
        handlelength=2.2,
    )

    # ── Panel B — Relative speedup ────────────────────────────────────────────

    # Parity line and colored zones
    all_ns = np.array(SIZES, dtype=float)
    ax_rel.axhline(1.0, color=MUTED, linewidth=1.5, linestyle="-", zorder=2)
    ax_rel.fill_between(
        [SIZES[0] * 0.85, SIZES[-1] * 1.15], [1.0, 1.0], [5.0, 5.0],
        color=GREEN, alpha=0.07, zorder=1,
    )
    ax_rel.fill_between(
        [SIZES[0] * 0.85, SIZES[-1] * 1.15], [0.0, 0.0], [1.0, 1.0],
        color=RED, alpha=0.07, zorder=1,
    )

    # Zone labels
    ax_rel.text(
        SIZES[0] * 0.92, 1.06, "gridr faster  ↑",
        fontsize=8.5, color=GREEN, alpha=0.8,
        fontfamily="monospace", fontweight="bold",
    )
    ax_rel.text(
        SIZES[0] * 0.92, 0.94, "scipy faster  ↓",
        fontsize=8.5, color=RED, alpha=0.8,
        fontfamily="monospace", fontweight="bold",
        va="top",
    )

    for method in methods:
        color = METHOD_COLORS[method]
        ns, ratios = _ratio_series(df, method)
        #print(ns, ratios)
        if len(ns) == 0:
            continue

        ax_rel.plot(
            ns, ratios,
            color=color, linewidth=2.2,
            marker="D", markersize=7,
            markeredgewidth=1.5, markeredgecolor=BG,
            zorder=4, label=METHOD_LABELS[method],
        )

        # Annotate the final point with the ratio value
        ax_rel.annotate(
            f"×{ratios[-1]:.2f}",
            xy=(ns[-1], ratios[-1]),
            xytext=(8, 0),
            textcoords="offset points",
            fontsize=8, color=color,
            va="center", fontfamily="monospace",
            fontweight="bold",
        )

    ax_rel.set_xscale("log")
    ax_rel.set_xlim(SIZES[0] * 0.85, SIZES[-1] * 1.4)

    # Y-axis: ratio in a sensible range
    all_ratios = []
    for m in methods:
        _, r = _ratio_series(df, m)
        all_ratios.extend(r.tolist())
    if all_ratios:
        ymin = min(0.5, min(all_ratios) * 0.85)
        ymax = max(1.5, max(all_ratios) * 1.15)
        ax_rel.set_ylim(ymin, ymax)

    ax_rel.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda y, _: f"×{y:.2f}")
    )
    ax_rel.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"{int(x):,}")
    )
    ax_rel.set_xticks(SIZES)
    ax_rel.set_xticklabels([f"{n:,}" for n in SIZES], fontsize=8.5)
    ax_rel.set_xlabel("Grid size n", fontsize=10, labelpad=8)
    ax_rel.set_ylabel("Speedup ratio  scipy / gridr", fontsize=10, labelpad=8)
    ax_rel.set_title(
        "Relative speedup  scipy / gridr",
        fontsize=12, fontweight="bold",
        color=TEXT, pad=14,
    )
    ax_rel.legend(
        loc="upper right",
        frameon=True,
        framealpha=0.25,
        facecolor=SURFACE,
        edgecolor=BORDER,
        labelcolor=TEXT,
        fontsize=8.5,
    )

    # ── Global title & footer ─────────────────────────────────────────────────
    fig.suptitle(
        "gridr  vs  scipy.map_coordinates  ·  Scalability",
        fontsize=15, fontweight="bold",
        color=TEXT, y=0.98,
        fontfamily="monospace",
    )
    commit = df["git_commit"].values[0]
    lib_version = df['lib_version'].values[0]
    fig.text(
        0.5, 0.005,
        f"AMD EPYC 7713  ·  Python 3.11.10  ·  gridr {lib_version}  ·  commit {commit}"
        "   ·   20 rounds, warmup=3, OMP_NUM_THREADS=1"
        "   ·   shaded band = IQR (Q1–Q3)",
        ha="center", va="bottom",
        fontsize=7.5, color=MUTED,
        fontfamily="monospace",
    )

    # ── Save ──────────────────────────────────────────────────────────────────
    if output is not None:
        out = Path(output)
        fig.savefig(
            out,
            format=fmt,
            dpi=dpi,
            bbox_inches="tight",
            facecolor=BG,
        )
        tag = f"{dpi} dpi" if fmt == "png" else "vector"
        print(f"✓ {out}  ({tag})")
    #return fig




In [ ]:
#FILE = Path("../results/pytest/Linux-CPython-3.11-64bit") / "20260610_120303_0001_v0.5.2_trex083.sis.cnes.fr.csv"
FILE =  Path("../results/pytest/Linux-CPython-3.11-64bit") / "20260610_132425_0002_v0.6.0.dev0_trex009.sis.cnes.fr.csv"
df = pd.read_csv(FILE, sep=";")


In [ ]:
df_scipy = df[ ~np.isnan(df['n']) ]

In [ ]:
df_scipy['method'] = [v.split('-')[1] for v in df_scipy['name'].values]
def translate_adapter(s):
    if 'Gridr' in s:
        return 'gridr'
    else:
        return 'scipy'
df_scipy['adapter'] = [translate_adapter(v) for v in df_scipy['adapter'].values]

In [ ]:
plot_scalability(
    df = df_scipy,
    output="./benchmark_scipy_v0.6.0.dev0.png"
)